# Chuẩn bị B1-shuf × 3 seed (40/80)

Chỉ sinh prompt/audit; không inference, Joern, CodeT5, commit hay OSF. Ghép cặp với B1 hiện có; chỉ thay graph. Target thiếu CPG giữ nguyên B1, không lấy donor và được gắn cờ.


In [ ]:
from pathlib import Path
import sys, json, importlib, os
PROJECT_ROOT_OVERRIDE = None  # Windows: r'D:\JUNIOR\LabThayTho\GRACE-summary'; WSL: '/mnt/d/JUNIOR/LabThayTho/GRACE-summary'
candidates = [Path(PROJECT_ROOT_OVERRIDE)] if PROJECT_ROOT_OVERRIDE else [Path.cwd(), *Path.cwd().parents]
ROOT = next((p.resolve() for p in candidates if (p/'tier1_prompt_tools/prepare_cached.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Đặt PROJECT_ROOT_OVERRIDE thành thư mục GRACE-summary')
os.environ['GRACE_SUMMARY_ROOT'] = str(ROOT)
ARM_ROOT = ROOT/'08_Arm_B1_Shuf'
BASELINE = ROOT/'01_Author_Baseline_Unified'
B1_ROOT = ROOT/'03_Arm_B1_Graph_Only/output/bundles'
B2_ROOT = BASELINE/'output/local_full/bundles_recovered_joern_final_20261031'
FROZEN_ROOT = BASELINE/'output/research_freeze_20261101'
OUT_ROOT = ARM_ROOT/'output/bundles_validation_v1'
# Mặc định chỉ validation. Muốn cả test: thêm ('devign','test'), ('reveal','test') và dùng OUT_ROOT mới.
RUNS = [('devign','validation'), ('reveal','validation')]
SEEDS = (42,43,44)
LOCAL_FILES_ONLY = True  # Đổi False nếu tokenizer chưa có cache; chỉ tải tokenizer, không tải LLM.
sys.path.insert(0, str(ARM_ROOT))
import prepare_shuf
importlib.reload(prepare_shuf)
print('Project:', ROOT, '\nOutput:', OUT_ROOT)


## Kiểm thử offline trước khi xuất

Dùng môi trường hiện có. Nếu thiếu dependencies: cài numpy, pytest, transformers==4.57.1 và sentencepiece vào môi trường đó. Bộ test chỉ dùng dữ liệu tổng hợp, không tải model.


In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pytest', str(ARM_ROOT/'test_prepare_shuf.py'), '-q'], check=True)


## Nạp đúng tokenizer

Không nạp encoder/LLM. Toàn bộ donor CPG được đọc một lần để kiểm hash và lập chỉ mục; graph lớn vẫn nằm trên đĩa để tránh giữ toàn bộ kho trong RAM.


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    'Qwen/Qwen3-Coder-30B-A3B-Instruct',
    revision=prepare_shuf.REVISION, trust_remote_code=False,
    local_files_only=LOCAL_FILES_ONLY,
)
assert tokenizer.chat_template
print('Tokenizer revision:', prepare_shuf.REVISION)


## Sinh prompt (CPU; có thể mất thời gian đọc/hash kho lớn)

Dừng khi input/hash/scaffold khác tham chiếu. Không ghi đè OUT_ROOT đã có dữ liệu: chọn tên output mới nếu chạy lại. Không coi ZIP là notebook inference; chưa có runtime B1-shuf.


In [ ]:
archive = prepare_shuf.prepare(
    b1_root=B1_ROOT, b2_root=B2_ROOT, frozen_root=FROZEN_ROOT,
    out_root=OUT_ROOT, tokenizer=tokenizer, runs=RUNS, seeds=SEEDS,
)
print('ZIP:', archive)


In [ ]:
for dataset, split in RUNS:
    folder = OUT_ROOT/'audit'/dataset/split
    report = json.loads((folder/'audit_summary.json').read_text(encoding='utf-8'))
    print(dataset, split, json.dumps(report, indent=2, ensure_ascii=False))
    sample = list(prepare_shuf.rows(folder/'manual_sample.jsonl'))
    print('Manual sample:', len(sample), folder/'manual_sample.jsonl')
    assert len(sample) == min(30, sum(r['n'] for r in report.values()))
    print(sample[0]['prompt'][:1800] if sample else 'Empty sample')


## Việc bạn làm sau khi chạy

Hai người đọc manual_sample.jsonl, ghi tên/nhận xét; đối chiếu mức nới ghép cỡ và tỷ lệ graph thiếu. Nếu graph thiếu >5%, xử lý trước khi diễn giải. Dùng PREREG_B1_SHUF_DRAFT.md để ghi thay đổi 40/80, scaffold B1 và chính sách graph thiếu. Không đăng ký ngược ngày; chỉ sau audit/smoke test backend đạt mới khóa giao thức cho lượt test mới. Không chạy inference từ notebook này.
